# HW3.proof_walk — Appendix A line-by-line, numerically audited

**Source.** [`PAPER-ARXIV.md`](../../../../PAPER-ARXIV.md) Appendix A (proof of Theorem 1, partition-conditional binary-entropy bracket).

**Doctrine.** Every algebraic step gets a numeric witness on a 5 000-point grid; if you can't certify the step in `float64`, you haven't understood it. We *do not* re-derive `hbin`, `hbin_inverse`, `upper_HR`, or `lower_Fano` — HW3 already did. We only *use* them.

## Setup

Resolve `onboarding/projects/` on `sys.path`, attach the reflection log, and import the canonical references (used only for spot-checks; the student version derives its own implementations).

In [ ]:
import os, sys
from pathlib import Path
_here = Path(os.getcwd()).resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'shared' / '__init__.py').exists() and _p.name == 'projects':
        _PROJECTS = _p
        break
else:
    raise RuntimeError('could not locate onboarding/projects/')
_REPO = _PROJECTS.parent.parent
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless for CI
import matplotlib.pyplot as plt

from onboarding.projects import reflect
reflect.start(notebook='hw3.proof_walk', store=_PROJECTS / '.reflection.jsonl')
print(f'[setup] projects root: {_PROJECTS}')


In [ ]:
# Re-import the bracket primitives from HW3-solution scope.
from math import log2
import numpy as np

def hbin(p):
    if p <= 0 or p >= 1: return 0.0
    return -(p*log2(p) + (1-p)*log2(1-p))

def hbin_inverse(h, tol=1e-12):
    if h <= 0: return 0.0
    if h >= 1: return 0.5
    lo, hi = 0.0, 0.5
    while hi - lo > tol:
        mid = 0.5 * (lo + hi)
        if hbin(mid) < h: lo = mid
        else:             hi = mid
    return 0.5 * (lo + hi)

def upper_HR(h):  return 0.5 * h
def lower_Fano(h): return hbin_inverse(h)


## A.1 — Setup

Fix binary $Y \in \{0, 1\}$ and a finite measurable partition $\Pi = \{C_1, \ldots, C_m\}$ of the input. Write $q_C = \Pr(C)$ and $p_C = \Pr(Y=1 \mid C)$. The **per-cell Bayes error** is $e_C = \min(p_C, 1-p_C) \in [0, 1/2]$. The **partition-conditional Bayes error** is

$$\varepsilon^*_\Pi = \sum_C q_C\, e_C, \qquad H(Y\mid\Pi) = \sum_C q_C\, H_\mathrm{bin}(e_C).$$

(Both sums use the **binarised** $e_C \in [0,1/2]$, on which $H_\mathrm{bin}$ is monotone increasing.)

In [ ]:
rng = np.random.default_rng(20251115)
N = 5000
M = 8                          # 8 cells per random partition
qs = rng.dirichlet(np.ones(M), size=N)            # (N, M) cell masses
es = rng.uniform(0.0, 0.5, size=(N, M))           # (N, M) per-cell e_C
Hbs = np.vectorize(hbin)(es)                       # (N, M) H_bin(e_C)
Hpi = (qs * Hbs).sum(axis=1)                       # (N,) H(Y|\Pi)
eps = (qs * es ).sum(axis=1)                       # (N,) \eps_\Pi
print(f'A.1 setup OK: N={N} random partitions, M={M} cells each')
print(f'  H(Y|\\Pi) range: [{Hpi.min():.4f}, {Hpi.max():.4f}]')
print(f'  \\eps_\\Pi   range: [{eps.min():.4f}, {eps.max():.4f}]')


## A.2 — Hellman–Raviv applied **per cell** (Lemma A.2)

For each cell, $e_C \le \tfrac{1}{2} H_\mathrm{bin}(e_C)$ (HR 1970, restated). This is *not* Jensen — it's pointwise concavity comparison of the chord through $(0, 0)$ and $(1/2, 1/2)$ against the entropy curve.

In [ ]:
pointwise_HR = es <= 0.5 * Hbs + 1e-12
assert pointwise_HR.all(), f'A.2: HR fails pointwise on {(~pointwise_HR).sum()} (cell, partition) pairs'
slack_HR = 0.5 * Hbs - es
print(f'A.2 OK: pointwise HR holds on all {pointwise_HR.size} cells')
print(f'  worst slack 0.5\u00b7H_bin(e) - e = {slack_HR.max():.4f}  (peak near e = 0)')


## A.3 — Sum over cells (linearity of expectation)

Multiply A.2 by $q_C \ge 0$ and sum:

$$\sum_C q_C e_C \;\le\; \sum_C q_C \cdot \tfrac{1}{2} H_\mathrm{bin}(e_C) \;=\; \tfrac{1}{2} H(Y\mid\Pi).$$

This is **the upper endpoint of Theorem 1**. **No Jensen** is invoked — only linearity. The slack on the left is *additive* across cells (a useful audit when refining).

In [ ]:
upper_holds = eps <= 0.5 * Hpi + 1e-12
assert upper_holds.all(), f'A.3: upper Theorem-1 fails on {(~upper_holds).sum()}/{N} partitions'
upper_slack = 0.5 * Hpi - eps
print(f'A.3 OK: \u03b5* \u2264 H/2 on all {N} partitions')
print(f'  worst upper slack: {upper_slack.max():.4f}')


## A.4 — Optional Jensen route (Lemma A.4)

An *alternate* (looser) derivation of $\varepsilon^* \le H/2$ uses Jensen on the concave map $e \mapsto \tfrac{1}{2} H_\mathrm{bin}(e)$:

$$\sum_C q_C \cdot \tfrac{1}{2} H_\mathrm{bin}(e_C) \;\le\; \tfrac{1}{2} H_\mathrm{bin}\!\big(\sum_C q_C e_C\big) = \tfrac{1}{2} H_\mathrm{bin}(\varepsilon^*).$$

This gives $\varepsilon^* \le \tfrac{1}{2} H_\mathrm{bin}(\varepsilon^*)$, equivalent to $\varepsilon^* \le 1/2$, which is trivial — confirming **A.3 is the right route** and Jensen here is a strict pedagogical detour.

In [ ]:
jensen_RHS = 0.5 * np.vectorize(hbin)(eps)
jensen_LHS = 0.5 * Hpi
assert (jensen_LHS <= jensen_RHS + 1e-12).all(), 'A.4: Jensen on concave H_bin failed'
print(f'A.4 OK: 0.5\u00b7H(Y|\u03a0) \u2264 0.5\u00b7H_bin(\u03b5*) on all {N} partitions (Jensen, looser bound)')


## A.5 — Fano applied **per cell** (Lemma A.5)

For each cell, $H_\mathrm{bin}(e_C) \le H(Y\mid C)$. For binary $Y$ this is **equality** by construction ($H(Y\mid C) = H_\mathrm{bin}(p_C) = H_\mathrm{bin}(e_C)$, since $H_\mathrm{bin}$ is symmetric). Inverting on the $[0, 1/2]$ branch (where $H_\mathrm{bin}$ is strictly monotone increasing): $e_C = H_\mathrm{bin}^{-1}(H_\mathrm{bin}(e_C))$.

In [ ]:
round_trip = np.vectorize(lambda e: hbin_inverse(hbin(e)))(es)
err = np.abs(round_trip - es).max()
assert err < 1e-9, f'A.5: H_bin^{{-1}} \u2218 H_bin round-trip off by {err}'
print(f'A.5 OK: pointwise H_bin^{{-1}}\u2218H_bin = id, max abs err {err:.2e}')


## A.6 — Convexity of $H_\mathrm{bin}^{-1}$ on $[0, 1]$

$H_\mathrm{bin}: [0, 1/2] \to [0, 1]$ is **concave and increasing**, so its inverse $H_\mathrm{bin}^{-1}: [0, 1] \to [0, 1/2]$ is **convex and increasing** (slope $1/H_\mathrm{bin}'(g(y))$ grows as $H_\mathrm{bin}'$ shrinks toward $y \to 1$). This is the **only** non-trivial analytic fact in the proof; we audit it numerically. *Caveat lector*: it is tempting to write "concave" by reflex — the entropy is concave — but the inverse map flips that.

In [ ]:
hs_grid = np.linspace(0.01, 0.99, 401)
inv_grid = np.array([hbin_inverse(h) for h in hs_grid])
# convexity test: second differences should be >= 0 (up to discretisation noise)
d2 = inv_grid[2:] - 2*inv_grid[1:-1] + inv_grid[:-2]
convexity_violations = (d2 < -1e-8).sum()
assert convexity_violations == 0, f'A.6: H_bin^{{-1}} convexity violated at {convexity_violations} grid points'
print(f'A.6 OK: H_bin^{{-1}} convex on 401-point grid (min d² = {d2.min():.2e})')


## A.7 — Jensen on the lower side $\Rightarrow$ Theorem 1 lower (Lemma A.7)

Combining A.5 (pointwise round-trip) and A.6 (convexity of $H_\mathrm{bin}^{-1}$), Jensen for a **convex** map gives $\sum q_C g(x_C) \ge g(\sum q_C x_C)$, hence

$$\varepsilon^* = \sum_C q_C e_C = \sum_C q_C\, H_\mathrm{bin}^{-1}(H_\mathrm{bin}(e_C)) \;\ge\; H_\mathrm{bin}^{-1}\!\Big(\sum_C q_C H_\mathrm{bin}(e_C)\Big) = H_\mathrm{bin}^{-1}(H(Y\mid\Pi)).$$

This is **the lower endpoint of Theorem 1**. The Jensen direction here is opposite to A.4 precisely because the outer map is convex (A.6) rather than concave (A.4).

In [ ]:
lower_holds = eps + 1e-9 >= np.vectorize(hbin_inverse)(Hpi)
assert lower_holds.all(), f'A.7: lower Theorem-1 fails on {(~lower_holds).sum()}/{N} partitions'
lower_slack = eps - np.vectorize(hbin_inverse)(Hpi)
print(f'A.7 OK: \u03b5* \u2265 H_bin^{{-1}}(H) on all {N} partitions')
print(f'  worst lower slack: {lower_slack.max():.4f}')


## A.summary — bracket holds on all 5000 random partitions

We have certified, numerically and step-by-step, every algebraic move of Appendix A on 5 000 random 8-cell partitions, with **zero failures** on either endpoint. The total bracket width $w(\Pi) = \tfrac{1}{2} H(Y\mid\Pi) - H_\mathrm{bin}^{-1}(H(Y\mid\Pi))$ is bounded above by the universal $w^\star \approx 0.1610$ at $H \approx H_\mathrm{bin}(1/5)$ (HW3.Q4).

In [ ]:
bracket_width = 0.5 * Hpi - np.vectorize(hbin_inverse)(Hpi)
w_star = 0.1610
assert bracket_width.max() <= w_star + 1e-3, f'A.summary: empirical bracket width {bracket_width.max():.4f} exceeds w* = {w_star}'
print(f'[GATE OK] Appendix A audited on {N} partitions; max bracket width {bracket_width.max():.4f} \u2264 w* = {w_star}')
reflect.log('Q3.proof_walk', f'Appendix A.1-A.7 numerically audited on {N} partitions; both endpoints hold pointwise', 'HIGH')
